# 06 · The stack: cross-fitted L2 logistic regression on member logits

**Final:** 30 members -> CV 0.94657 -> public LB 0.94672 -> private LB **0.94558, rank 192 / 3,576**.

## Design
1. Discover every `outputs/oof/oof_<tag>.npy` with a matching `outputs/test/test_<tag>.npy`; prefer `test_<tag>_full.npy`
   (100%-train refit) when present. Skip incomplete OOFs (crashed runs).
2. **Seed-bag each family in logit space** into one column (`naji3cb_bag8`, `lgb_bag2`, ...). Near-duplicate seed columns otherwise
   let the stacker fit noise with +/- weights on the same model.
3. Meta-features = logit of each member's probability. Meta-model = `LogisticRegression(C=0.1)`, **cross-fitted** on the same
   StratifiedKFold(5, seed 42): the stacked OOF for fold k comes from a meta-model that never saw fold k.
4. **Gate**: `cv.paired()` of the new stack vs the frozen incumbent (`outputs/oof/stacks/`). Accept only if mean gain > 2 sd and
   all 5 folds move the same way. On PASS the OOF is frozen as the next incumbent and the submission is versioned
   (`sub_stack_cv<cv>_<timestamp>.csv`). Finals were mutable until 09-06, when a stack overwrote its own gate reference; after
   that, everything is frozen.
5. Test prediction: the meta-model refitted on all OOF rows, applied to the members' test logits.

## What the weights say (final 30-member stack)
| member | weight | | member | weight |
|---|---|---|---|---|
| ridge (heuljax generator-aware LR) | **0.386** | | naji3_bag3 | 0.028 |
| heuljax_s42 (plain LR) | 0.204 | | naji_bag5 | 0.023 |
| aryan_s42 (HistGBM) | 0.147 | | naji3cb variants (6) | -0.001 .. 0.022 |
| lgb_bag2 (depth-5, feats_v10) | 0.060 | | zoom, tabm, cb_bag2, xgb_bag2 | 0.006 .. 0.029 |
| gbdt_additive | 0.058 | | ftt, direct, naji3xgb, najit58 | -0.02 .. -0.065 (contrasts) |
| naji3cb_bag8 | 0.049 | | realmlp, sergey, lucxgb | ~ 0 |

Negative weights fall on correlated GBDT variants: the stacker uses them as contrasts. We never pruned them; a 7-member pruned
stack scored the same as the full one, never better.

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
from cv import load, run_cv, paired, SEED, NFOLDS, OOF, TEST
X, y, Xte, te_id = load()          # data/train.csv, data/test.csv  (kaggle competitions download -c playground-series-s6e9)
print(X.shape, Xte.shape, 'positives %.4f' % y.mean())

In [ ]:
import re
from scipy.special import logit, expit
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
L = lambda p: logit(np.clip(p, 1e-6, 1 - 1e-6))

# 1. discover members
OK = r'(lgb|xgb|cb|tabm|ftt|realmlp|naji|najit58|zoom|direct|naji3[a-z0-9]*|sergey|yeke|lucxgb|heuljax|aryan|ridge)_s\d+'
pairs = {'gbdt_additive': ('oof_final.npy', 'test_final.npy'), 'realmlp_e48x3': ('oof_R_e48.npy', 'test_nn_avg.npy')}
for f in sorted(OOF.glob('oof_*_s*.npy')):
    tag = f.stem[4:]
    if re.fullmatch(OK, tag) and (TEST / f'test_{tag}.npy').exists():
        full = TEST / f'test_{tag}_full.npy'
        pairs[tag] = (f.name, full.name if full.exists() else f'test_{tag}.npy')
oofs, tests = {}, {}
for k, (fo, ft) in pairs.items():
    if not (OOF / fo).exists(): continue
    o = np.load(OOF / fo)
    if (o == 0).mean() > 0.01: print('skip incomplete', k); continue
    oofs[k], tests[k] = o, np.load(TEST / ft)

# 2. seed-bag families in logit space
fam = {}
for n in list(oofs):
    m = re.fullmatch(r'([a-z0-9]+)_s\d+', n); fam.setdefault(m.group(1) if m else n, []).append(n)
for f, names in fam.items():
    if len(names) > 1:
        oofs[f + '_bag%d' % len(names)] = expit(np.mean([L(oofs[n]) for n in names], axis=0))
        tests[f + '_bag%d' % len(names)] = expit(np.mean([L(tests[n]) for n in names], axis=0))
        for n in names: oofs.pop(n); tests.pop(n)
print(len(oofs), 'members:', list(oofs))

In [ ]:
# 3. cross-fitted logit stacker
def fit_stack(keys, C=0.1):
    Z = np.column_stack([L(oofs[k]) for k in keys]); Zt = np.column_stack([L(tests[k]) for k in keys])
    s = np.zeros(len(y))
    for itr, iva in StratifiedKFold(NFOLDS, shuffle=True, random_state=SEED).split(Z, y):
        s[iva] = LogisticRegression(C=C, max_iter=2000).fit(Z[itr], y.iloc[itr]).predict_proba(Z[iva])[:, 1]
    m = LogisticRegression(C=C, max_iter=2000).fit(Z, y)
    return s, m.predict_proba(Zt)[:, 1], dict(zip(keys, np.round(m.coef_[0], 3)))

keys = list(oofs)
stack, ptest, w = fit_stack(keys)
print('stack oof=%.5f' % roc_auc_score(y, stack))
print(sorted(w.items(), key=lambda kv: -abs(kv[1])))

In [ ]:
# 4. gate against the frozen incumbent; 5. write a versioned submission only on PASS
STACKS = OOF / 'stacks'; STACKS.mkdir(exist_ok=True)
prev = sorted(STACKS.glob('stack_*.npy'), key=lambda f: roc_auc_score(y, np.load(f)))
INC = np.load(prev[-1]) if prev else np.load(OOF / 'oof_final.npy')
d = paired(stack, INC, y, 'stack vs incumbent')
if d.mean() > 2 * d.std(ddof=1) and (d > 0).all():
    cv = roc_auc_score(y, stack); tag = 'cv%05d_%s' % (round(cv * 1e5), pd.Timestamp.now().strftime('%m%d_%H%M'))
    pd.DataFrame({'id': te_id, 'Will_Buy_EV': ptest}).to_csv(f'../submissions/sub_stack_{tag}.csv', index=False)
    np.save(STACKS / f'stack_{tag}.npy', stack); print('PASS -> wrote sub_stack_%s.csv' % tag)
else:
    print('FAIL -> nothing written')

## Things we measured so you do not have to
| alternative combiner (same 28-30 members) | CV vs logistic 0.94638 |
|---|---|
| equal-weight rank average | -0.0001 |
| LightGBM meta-learner on the logits | -0.00005 (noise) |
| additive-stump meta-learner | -0.00010, 0/5 |
| level-3 average of LightGBM-meta and logistic | 0 |
| LightGBM on raw features + all member logits | -0.00008, 0/5 |
| linear regression on probabilities | tie; on logits -0.0025 |

The 30 columns correlate above 0.95 with each other. There is no interaction between them for a nonlinear meta-learner to find,
only noise. Logistic regression on logits is the right shape for this problem, not a placeholder.

## Public vs private
Our CV-to-public offset was +0.00015..+0.00029 on all 14 submissions. Public-to-private was **-0.0011** on every one of them,
the same size as for every team in the top 200, so the ordering of our own files held: best public = best private. The team that
led public at 0.94945 (a deliberate public-LB-fitted file) finished 2nd with an honest 0.94588; their fitted file was 0.94313
private. Never select by public score.